## Iceburg Table Connection

### Load Jar Files

In [1]:
import os
import sys
# 1. Set PYSPARK_SUBMIT_ARGS to match your working batch file launcher
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    '--conf spark.driver.extraClassPath="C:/data/spark/jars/iceberg-spark-runtime-4.0_2.13-1.10.0.jar" '
    "pyspark-shell"
)

# 2. Ensure Python paths align for the worker processes
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

### Spark Connection

In [7]:
from pyspark.sql import SparkSession

BUCKET_NAME = "iceberg"

LOCAL_WAREHOUSE_PATH = f"s3a://{BUCKET_NAME}/iceberg_warehouse"
STG_WAREHOUSE_PATH   = f"s3a://{BUCKET_NAME}/iceberg_staging_warehouse"
RPT_WAREHOUSE_PATH   = f"s3a://{BUCKET_NAME}/WideWorldImportersDW"

CATALOG_NAME = "local"
STG_CATALOG_NAME = "staging"
WH_CATALOG_NAME = "reporting"

spark = SparkSession.builder \
    .appName("Iceberg Setup") \
    .config("spark.jars.packages", 
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.2,"
            "org.apache.hadoop:hadoop-aws:3.3.4,"
            "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{CATALOG_NAME}.warehouse", LOCAL_WAREHOUSE_PATH) \
    \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{STG_CATALOG_NAME}.warehouse", STG_WAREHOUSE_PATH) \
    \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}", "org.apache.iceberg.spark.SparkCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog") \
    .config(f"spark.sql.catalog.{WH_CATALOG_NAME}.warehouse", RPT_WAREHOUSE_PATH) \
    \
    .config("spark.hadoop.fs.s3a.endpoint", "http://127.0.0.1:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .getOrCreate()

spark.catalog.setCurrentCatalog(WH_CATALOG_NAME)
spark.sql("CREATE NAMESPACE IF NOT EXISTS reporting.dimension")
spark.sql("CREATE NAMESPACE IF NOT EXISTS staging.Integration")

# Show available namespaces
namespaces = spark.sql("SHOW NAMESPACES IN reporting").collect()

for ns in namespaces:
    namespace = ns["namespace"]
    print(f"\nTables in Namespace: {namespace}")
    spark.sql(f"SHOW TABLES IN reporting.{namespace}").show(truncate=False)


Tables in Namespace: dimension
+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|dimension|dates    |false      |
+---------+---------+-----------+



## date dim Table

In [9]:
from pyspark.sql import functions as F

# Creates 5 rows with dummy = 1
df_dates = spark.range(0, (365*4) + 1) \
    .withColumn("date_key", F.date_add(F.lit("2023-01-01").cast("date"), F.col("id").cast("int"))) \
    .drop("id")

df_dates = df_dates \
    .withColumn("year", F.year("date_key")) \
    .withColumn("year_short", F.date_format("date_key", "yy")) \
    .withColumn("month", F.month("date_key")) \
    .withColumn("month_name", F.date_format("date_key", "MMMM")) \
    .withColumn("month_name_short", F.date_format("date_key", "MMM")) \
    .withColumn("day", F.dayofmonth("date_key")) \
    .withColumn("formatted_date", F.concat(F.dayofmonth("date_key"), (F.when(F.dayofmonth("date_key").isin(11, 12, 13), "th") \
                                                 .when(F.dayofmonth("date_key") % 10 == 1, "st") \
                                                 .when(F.dayofmonth("date_key") % 10 == 2, "nd") \
                                                 .when(F.dayofmonth("date_key") % 10 == 3, "rd") \
                                                 .otherwise("th")
                                                ) \
                                        ) \
                ) \
    .withColumn("day_name", F.date_format("date_key", "EEEE")) \
    .withColumn("day_name_short", F.date_format("date_key", "EEE")) \
    .withColumn("quarter", F.quarter("date_key")) \
    .withColumn("week_of_year", F.weekofyear("date_key")) \
    .withColumn("day_of_week", F.dayofweek("date_key")) \
    .withColumn("day_of_year", F.dayofyear("date_key")) \
    .withColumn("is_weekend", F.when(F.dayofweek("date_key").isin([1, 7]), True).otherwise(False)) \

df_dates = df_dates.withColumn("british_date_format", F.date_format("date_key", "dd/MM/yyyy")) \
    .withColumn("american_date_format", F.date_format("date_key", "MM/dd/yyyy")) \
    .withColumn("iso_date_format", F.date_format("date_key", "yyyy-MM-dd")) \

df_dates = df_dates.withColumn("yesterday_date", F.date_format(F.date_sub("date_key", 1), "yyyy-MM-dd")) \
    .withColumn("today_date", F.date_format("date_key", "yyyy-MM-dd")) \
    .withColumn("tomorrow_date", F.date_format(F.date_add("date_key", 1), "yyyy-MM-dd"))

df_dates = df_dates.withColumn("first_day_of_month", F.trunc("date_key", "MM")) \
        .withColumn("last_day_of_month", F.last_day("date_key"))

df_dates = df_dates.withColumn("first_day_of_year", F.trunc("date_key", "yyyy")) \
        .withColumn("last_day_of_year", F.make_date(F.year("date_key"), F.lit(12), F.lit(31)))

df_dates = df_dates.withColumn("dateyear", F.trunc("date_key", "yyyy")) \



# Chaining multiple properties correctly
df_dates.writeTo("reporting.dimension.dates") \
    .using("iceberg") \
    .partitionedBy("dateyear") \
    .tableProperty("comment", "dimension table for 4 Years Date") \
    .tableProperty("write.format.default", "parquet") \
    .createOrReplace()


In [8]:
spark.sql("DROP TABLE IF EXISTS reporting.dimension.dates PURGE")

DataFrame[]

In [12]:
# 1. Query the properties metadata table (Returns property key-value pairs)
# spark.sql("SELECT * FROM reporting.dimension.dates.properties").show(truncate=False)

# 2. Use SHOW TBLPROPERTIES to view all properties
spark.sql("SHOW TBLPROPERTIES reporting.dimension.dates").show(truncate=False)

# 3. View a specific property
spark.sql("SHOW TBLPROPERTIES reporting.dimension.dates ('comment')").show()

+-------------------------------+-------------------+
|key                            |value              |
+-------------------------------+-------------------+
|current-snapshot-id            |4944900413479295101|
|format                         |iceberg/parquet    |
|format-version                 |2                  |
|write.format.default           |parquet            |
|write.parquet.compression-codec|zstd               |
+-------------------------------+-------------------+

+-------+--------------------+
|    key|               value|
+-------+--------------------+
|comment|Table reporting.d...|
+-------+--------------------+



In [21]:
from pyspark.sql import functions as F
df_date_table = spark.table("reporting.dimension.dates")
df_date_table.show(10)
df_date_table.agg(F.min("date_key").alias("min_date"), F.max("date_key").alias("max_date")).show()

df_date_table.printSchema()



+----------+----+----------+-----+----------+----------------+---+--------------+---------+--------------+-------+------------+-----------+-----------+----------+-------------------+--------------------+---------------+--------------+----------+-------------+------------------+-----------------+-----------------+----------------+----------+
|  date_key|year|year_short|month|month_name|month_name_short|day|formatted_date| day_name|day_name_short|quarter|week_of_year|day_of_week|day_of_year|is_weekend|british_date_format|american_date_format|iso_date_format|yesterday_date|today_date|tomorrow_date|first_day_of_month|last_day_of_month|first_day_of_year|last_day_of_year|  dateyear|
+----------+----+----------+-----+----------+----------------+---+--------------+---------+--------------+-------+------------+-----------+-----------+----------+-------------------+--------------------+---------------+--------------+----------+-------------+------------------+-----------------+-----------------+

In [10]:
# Check if data exists via Spark SQL / Iceberg metadata
print(spark.table("reporting.dimension.dates").count())

# Inspect Iceberg data files metadata table directly
spark.sql("SELECT file_path, file_format, record_count FROM reporting.dimension.dates.files").show(truncate=False)

1461
+-------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+
|file_path                                                                                                                                        |file_format|record_count|
+-------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+
|s3a://iceberg/iceberg/WideWorldImportersDW/dimension/dates/data/dateyear=2025-01-01/00000-46-91ab94e1-8dd9-40f8-a650-55cc31f8aa84-0-00003.parquet|PARQUET    |365         |
|s3a://iceberg/iceberg/WideWorldImportersDW/dimension/dates/data/dateyear=2026-01-01/00000-46-91ab94e1-8dd9-40f8-a650-55cc31f8aa84-0-00004.parquet|PARQUET    |365         |
|s3a://iceberg/iceberg/WideWorldImportersDW/dimension/dates/data/dateyear=2024-01-01/00000-46-91ab94e1-8dd9-40f8-a650-55cc31f8aa84

In [11]:
# Check the S3 paths of the generated Parquet files
spark.sql("""
    SELECT file_path, file_format, record_count 
    FROM reporting.dimension.dates.files
""").show(truncate=False)

+-------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+
|file_path                                                                                                                                        |file_format|record_count|
+-------------------------------------------------------------------------------------------------------------------------------------------------+-----------+------------+
|s3a://iceberg/iceberg/WideWorldImportersDW/dimension/dates/data/dateyear=2025-01-01/00000-46-91ab94e1-8dd9-40f8-a650-55cc31f8aa84-0-00003.parquet|PARQUET    |365         |
|s3a://iceberg/iceberg/WideWorldImportersDW/dimension/dates/data/dateyear=2026-01-01/00000-46-91ab94e1-8dd9-40f8-a650-55cc31f8aa84-0-00004.parquet|PARQUET    |365         |
|s3a://iceberg/iceberg/WideWorldImportersDW/dimension/dates/data/dateyear=2024-01-01/00000-46-91ab94e1-8dd9-40f8-a650-55cc31f8aa84-0-00